In [1]:
import ast
import math
import pandas as pd
import osmnx as ox
from shapely.geometry import LineString
from shapely.ops import unary_union

DATA_FILE = "osm_segment_metadata.csv"
GRAPH_FILE = "bhubaneswar_drive.graphml"

print("Loading dataset...")
df = pd.read_csv(DATA_FILE)

print("Loading OSM graph...")
G = ox.load_graphml(GRAPH_FILE)


# =========================================================
# 1. Parse dataset OSM IDs
# =========================================================

def parse_osmid(value):

    if pd.isna(value):
        return []

    value = str(value).strip()

    if value.startswith("[") and value.endswith("]"):
        try:
            return [str(x) for x in ast.literal_eval(value)]
        except Exception:
            return []

    return [value]


df["osmid_list"] = df["osmid"].apply(parse_osmid)


# =========================================================
# 2. Build ONE physical geometry per OSM ID
# =========================================================

osm_geometry = {}

for u, v, k, data in G.edges(keys=True, data=True):

    osmid = data.get("osmid")

    if osmid is None:
        continue

    if isinstance(osmid, list):
        ids = [str(x) for x in osmid]
    else:
        ids = [str(osmid)]

    # Get geometry
    geometry = data.get("geometry")

    if geometry is None:

        u_data = G.nodes[u]
        v_data = G.nodes[v]

        geometry = LineString([
            (
                float(u_data["x"]),
                float(u_data["y"])
            ),
            (
                float(v_data["x"]),
                float(v_data["y"])
            )
        ])

    # IMPORTANT:
    # Store only ONE geometry for each physical OSM way.
    for oid in ids:

        if oid not in osm_geometry:
            osm_geometry[oid] = geometry


print("\nUnique physical OSM geometries:",
      len(osm_geometry))


# =========================================================
# 3. Project graph to metric CRS
# =========================================================

print("Projecting OSM graph...")

G_projected = ox.project_graph(G)

# Build projected geometry lookup again
osm_geometry_projected = {}

for u, v, k, data in G_projected.edges(
    keys=True,
    data=True
):

    osmid = data.get("osmid")

    if osmid is None:
        continue

    if isinstance(osmid, list):
        ids = [str(x) for x in osmid]
    else:
        ids = [str(osmid)]

    geometry = data.get("geometry")

    if geometry is None:

        u_data = G_projected.nodes[u]
        v_data = G_projected.nodes[v]

        geometry = LineString([
            (
                float(u_data["x"]),
                float(u_data["y"])
            ),
            (
                float(v_data["x"]),
                float(v_data["y"])
            )
        ])

    for oid in ids:

        if oid not in osm_geometry_projected:
            osm_geometry_projected[oid] = geometry


# =========================================================
# 4. Haversine
# =========================================================

def haversine(lat1, lon1, lat2, lon2):

    R = 6371000

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)

    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1)
        * math.cos(phi2)
        * math.sin(dlambda / 2) ** 2
    )

    return 2 * R * math.asin(math.sqrt(a))


# =========================================================
# 5. Validate each traffic segment
# =========================================================

results = []

for _, row in df.iterrows():

    segment_id = row["segment_id"]

    dataset_ids = row["osmid_list"]

    dataset_length = float(row["length_m"])

    dataset_lat = float(row["lat"])
    dataset_lon = float(row["lon"])

    geometries = []

    missing_ids = []

    for oid in dataset_ids:

        if oid in osm_geometry_projected:

            geometries.append(
                osm_geometry_projected[oid]
            )

        else:

            missing_ids.append(oid)


    if not geometries:

        results.append({
            "segment_id": segment_id,
            "matched": False,
            "dataset_length_m": dataset_length
        })

        continue


    # -----------------------------------------------------
    # Combine physical geometries
    # -----------------------------------------------------

    combined = unary_union(geometries)

    osm_length = combined.length

    length_error = abs(
        dataset_length - osm_length
    )

    length_error_pct = (
        length_error / dataset_length * 100
        if dataset_length > 0
        else None
    )


    # -----------------------------------------------------
    # Geographic centroid
    # -----------------------------------------------------

    centroid = combined.centroid

    # Projected centroid -> geographic coordinates
    # Transform using OSMnx utilities
    import geopandas as gpd

    centroid_gdf = gpd.GeoDataFrame(
        geometry=[centroid],
        crs=G_projected.graph["crs"]
    )

    centroid_geo = centroid_gdf.to_crs(
        "EPSG:4326"
    ).geometry.iloc[0]

    osm_lat = centroid_geo.y
    osm_lon = centroid_geo.x

    coordinate_distance = haversine(
        dataset_lat,
        dataset_lon,
        osm_lat,
        osm_lon
    )


    results.append({

        "segment_id": segment_id,

        "matched": True,

        "num_dataset_osm_ids": len(dataset_ids),

        "missing_osm_ids": len(missing_ids),

        "dataset_length_m": dataset_length,

        "osm_geometry_length_m": osm_length,

        "length_error_m": length_error,

        "length_error_pct": length_error_pct,

        "dataset_lat": dataset_lat,

        "dataset_lon": dataset_lon,

        "osm_centroid_lat": osm_lat,

        "osm_centroid_lon": osm_lon,

        "coordinate_distance_m":
            coordinate_distance,

        "matched_osm_ids":
            ", ".join(dataset_ids)

    })


# =========================================================
# 6. Results
# =========================================================

results_df = pd.DataFrame(results)

matched = results_df[
    results_df["matched"] == True
].copy()


print("\n========================================")
print("FINAL OSM GEOMETRY VALIDATION")
print("========================================")

print(
    "\nTotal segments:",
    len(results_df)
)

print(
    "Matched:",
    len(matched)
)

print(
    "Match rate:",
    f"{len(matched) / len(results_df) * 100:.2f}%"
)


# =========================================================
# LENGTH RESULTS
# =========================================================

print("\n========== LENGTH ==========")

print(
    "Mean error:",
    f"{matched['length_error_m'].mean():.2f} m"
)

print(
    "Median error:",
    f"{matched['length_error_m'].median():.2f} m"
)

print(
    "Mean percentage error:",
    f"{matched['length_error_pct'].mean():.2f}%"
)

print(
    "Median percentage error:",
    f"{matched['length_error_pct'].median():.2f}%"
)

print(
    "Within 5%:",
    f"{(matched['length_error_pct'] <= 5).sum()} / {len(matched)}"
)

print(
    "Within 10%:",
    f"{(matched['length_error_pct'] <= 10).sum()} / {len(matched)}"
)

print(
    "Within 20%:",
    f"{(matched['length_error_pct'] <= 20).sum()} / {len(matched)}"
)


# =========================================================
# COORDINATES
# =========================================================

print("\n========== COORDINATES ==========")

print(
    "Mean distance:",
    f"{matched['coordinate_distance_m'].mean():.2f} m"
)

print(
    "Median distance:",
    f"{matched['coordinate_distance_m'].median():.2f} m"
)

print(
    "Within 50 m:",
    f"{(matched['coordinate_distance_m'] <= 50).sum()} / {len(matched)}"
)

print(
    "Within 100 m:",
    f"{(matched['coordinate_distance_m'] <= 100).sum()} / {len(matched)}"
)

print(
    "Within 500 m:",
    f"{(matched['coordinate_distance_m'] <= 500).sum()} / {len(matched)}"
)


# =========================================================
# WORST MISMATCHES
# =========================================================

print("\n========== WORST LENGTH MISMATCHES ==========")

worst = matched.sort_values(
    "length_error_pct",
    ascending=False
).head(15)

print(
    worst[
        [
            "segment_id",
            "num_dataset_osm_ids",
            "dataset_length_m",
            "osm_geometry_length_m",
            "length_error_m",
            "length_error_pct",
            "matched_osm_ids"
        ]
    ].to_string(index=False)
)


# =========================================================
# SAVE
# =========================================================

output_file = "osm_geometry_validation.csv"

results_df.to_csv(
    output_file,
    index=False
)

print("\nSaved:", output_file)

Loading dataset...
Loading OSM graph...

Unique physical OSM geometries: 11337
Projecting OSM graph...

FINAL OSM GEOMETRY VALIDATION

Total segments: 700
Matched: 700
Match rate: 100.00%

========== LENGTH ==========
Mean error: 119.53 m
Median error: 59.20 m
Mean percentage error: 69.90%
Median percentage error: 50.58%
Within 5%: 165 / 700
Within 10%: 191 / 700
Within 20%: 235 / 700

========== COORDINATES ==========
Mean distance: 339.33 m
Median distance: 194.65 m
Within 50 m: 168 / 700
Within 100 m: 230 / 700
Within 500 m: 542 / 700

========== WORST LENGTH MISMATCHES ==========
 segment_id  num_dataset_osm_ids  dataset_length_m  osm_geometry_length_m  length_error_m  length_error_pct                  matched_osm_ids
        610                    1         72.761861            1217.305702     1144.543841       1572.999677                         26755741
        533                    1         85.487647            1271.804177     1186.316530       1387.705212                    

In [2]:
import duckdb
import pandas as pd

DB_PATH = r"C:\Users\Mohit sharma\Desktop\7th sem\new major project\geopulse\datasets\traffic_v1.duckdb"

con = duckdb.connect(DB_PATH)

# --------------------------------------------------
# 1. BASIC TIME RANGE
# --------------------------------------------------

print("\n========== 1. TIME RANGE ==========")

result = con.execute("""
    SELECT
        MIN(hour) AS start_time,
        MAX(hour) AS end_time,
        COUNT(DISTINCT hour) AS unique_hours,
        COUNT(DISTINCT segment_id) AS segments
    FROM traffic
""").fetchdf()

print(result.to_string(index=False))


# --------------------------------------------------
# 2. OBSERVATION COVERAGE
# --------------------------------------------------

print("\n========== 2. OBSERVATION COVERAGE ==========")

result = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN is_observed THEN 1 ELSE 0 END) AS observed_rows,
        SUM(CASE WHEN is_missing THEN 1 ELSE 0 END) AS missing_rows,
        ROUND(
            100.0 * SUM(CASE WHEN is_observed THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS observed_percentage
    FROM traffic
""").fetchdf()

print(result.to_string(index=False))


# --------------------------------------------------
# 3. HOURS PER SEGMENT
# --------------------------------------------------

print("\n========== 3. HOURS PER SEGMENT ==========")

result = con.execute("""
    SELECT
        MIN(hour_count) AS minimum_hours,
        MAX(hour_count) AS maximum_hours,
        ROUND(AVG(hour_count), 2) AS average_hours,
        MEDIAN(hour_count) AS median_hours
    FROM (
        SELECT
            segment_id,
            COUNT(*) FILTER (WHERE is_observed) AS hour_count
        FROM traffic
        GROUP BY segment_id
    )
""").fetchdf()

print(result.to_string(index=False))


# --------------------------------------------------
# 4. SEGMENTS WITH LOW COVERAGE
# --------------------------------------------------

print("\n========== 4. LOW-COVERAGE SEGMENTS ==========")

result = con.execute("""
    SELECT
        segment_id,
        COUNT(*) AS total_hours,
        SUM(CASE WHEN is_observed THEN 1 ELSE 0 END) AS observed_hours,
        ROUND(
            100.0 * SUM(CASE WHEN is_observed THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS coverage_percentage
    FROM traffic
    GROUP BY segment_id
    ORDER BY coverage_percentage ASC
    LIMIT 20
""").fetchdf()

print(result.to_string(index=False))


# --------------------------------------------------
# 5. CONTINUOUS TIME COVERAGE
# --------------------------------------------------

print("\n========== 5. SEGMENTS WITH MOST MISSING HOURS ==========")

result = con.execute("""
    SELECT
        segment_id,
        SUM(CASE WHEN is_missing THEN 1 ELSE 0 END) AS missing_hours,
        COUNT(*) AS total_hours,
        ROUND(
            100.0 * SUM(CASE WHEN is_missing THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS missing_percentage
    FROM traffic
    GROUP BY segment_id
    ORDER BY missing_hours DESC
    LIMIT 20
""").fetchdf()

print(result.to_string(index=False))


# --------------------------------------------------
# 6. HOURS BY DAY
# --------------------------------------------------

print("\n========== 6. DAILY COVERAGE ==========")

result = con.execute("""
    SELECT
        DATE_TRUNC('day', hour) AS date,
        COUNT(*) AS total_records,
        SUM(CASE WHEN is_observed THEN 1 ELSE 0 END) AS observed_records,
        COUNT(DISTINCT segment_id) FILTER (WHERE is_observed) AS active_segments
    FROM traffic
    GROUP BY DATE_TRUNC('day', hour)
    ORDER BY date
""").fetchdf()

print(result.to_string(index=False))


# --------------------------------------------------
# 7. DAY-OF-WEEK COVERAGE
# --------------------------------------------------

print("\n========== 7. DAY OF WEEK COVERAGE ==========")

result = con.execute("""
    SELECT
        day_of_week,
        COUNT(*) AS total_records,
        SUM(CASE WHEN is_observed THEN 1 ELSE 0 END) AS observed_records
    FROM (
        SELECT
            EXTRACT(DOW FROM hour) AS day_of_week,
            is_observed
        FROM traffic
    )
    GROUP BY day_of_week
    ORDER BY day_of_week
""").fetchdf()

print(result.to_string(index=False))


# --------------------------------------------------
# 8. HOUR-OF-DAY COVERAGE
# --------------------------------------------------

print("\n========== 8. HOUR OF DAY COVERAGE ==========")

result = con.execute("""
    SELECT
        hour_of_day,
        COUNT(*) AS total_records,
        SUM(CASE WHEN is_observed THEN 1 ELSE 0 END) AS observed_records
    FROM traffic
    GROUP BY hour_of_day
    ORDER BY hour_of_day
""").fetchdf()

print(result.to_string(index=False))


con.close()

print("\n========== STEP 1 COMPLETE ==========")


========== 1. TIME RANGE ==========
               start_time                  end_time  unique_hours  segments
2026-01-01 00:30:00+05:30 2026-01-17 09:30:00+05:30           394       700

========== 2. OBSERVATION COVERAGE ==========
 total_rows  observed_rows  missing_rows  observed_percentage
     275800       246399.0       29401.0                89.34

========== 3. HOURS PER SEGMENT ==========
 minimum_hours  maximum_hours  average_hours  median_hours
           345            359          352.0         351.5

========== 4. LOW-COVERAGE SEGMENTS ==========
 segment_id  total_hours  observed_hours  coverage_percentage
        351          394           345.0                87.56
        356          394           345.0                87.56
        355          394           345.0                87.56
        359          394           345.0                87.56
        358          394           345.0                87.56
        352          394           345.0                87

BinderException: Binder Error: Referenced column "hour_of_day" not found in FROM clause!
Candidate bindings: "hour", "osmid", "confidence", "road_name", "road_type"

LINE 7:     GROUP BY hour_of_day
                     ^

In [4]:
import duckdb

DB_PATH = r"C:\Users\Mohit sharma\Desktop\7th sem\new major project\geopulse\datasets\traffic_v1.duckdb"

con = duckdb.connect(DB_PATH)

print("\n========== DAILY COVERAGE ==========")

print(con.execute("""
SELECT
    DATE_TRUNC('day', hour) AS date,
    COUNT(*) AS total_records,
    SUM(CASE WHEN is_observed THEN 1 ELSE 0 END) AS observed_records,
    COUNT(DISTINCT segment_id) FILTER (WHERE is_observed) AS active_segments
FROM traffic
GROUP BY DATE_TRUNC('day', hour)
ORDER BY date
""").fetchdf().to_string(index=False))


print("\n========== DAY OF WEEK COVERAGE ==========")

print(con.execute("""
SELECT
    EXTRACT(DOW FROM hour) AS day_of_week,
    COUNT(*) AS total_records,
    SUM(CASE WHEN is_observed THEN 1 ELSE 0 END) AS observed_records
FROM traffic
GROUP BY EXTRACT(DOW FROM hour)
ORDER BY day_of_week
""").fetchdf().to_string(index=False))


print("\n========== HOUR OF DAY COVERAGE ==========")

print(con.execute("""
SELECT
    EXTRACT(HOUR FROM hour) AS hour_of_day,
    COUNT(*) AS total_records,
    SUM(CASE WHEN is_observed THEN 1 ELSE 0 END) AS observed_records
FROM traffic
GROUP BY EXTRACT(HOUR FROM hour)
ORDER BY hour_of_day
""").fetchdf().to_string(index=False))

con.close()


========== DAILY COVERAGE ==========
                     date  total_records  observed_records  active_segments
2026-01-01 00:00:00+05:30          16800             350.0              350
2026-01-02 00:00:00+05:30          16800            4200.0              350
2026-01-03 00:00:00+05:30          16800           16450.0              700
2026-01-04 00:00:00+05:30          16800           16800.0              700
2026-01-05 00:00:00+05:30          16800           16800.0              700
2026-01-06 00:00:00+05:30          16800           16800.0              700
2026-01-07 00:00:00+05:30          16800           16800.0              700
2026-01-08 00:00:00+05:30          16800           16800.0              700
2026-01-09 00:00:00+05:30          16800           16800.0              700
2026-01-10 00:00:00+05:30          16800           16800.0              700
2026-01-11 00:00:00+05:30          16800           16800.0              700
2026-01-12 00:00:00+05:30          16800          